# Project 2 · Company financials from Yahoo Finance (yfinance)

For the company dashboard on the website. Fetches the 50 source companies that have a ticker, **one ticker at a time** with a 3-second pause (about 3–4 minutes), caches each ticker for the day, and writes one snapshot file. The web page cannot call Yahoo itself, so it shows this snapshot with its date and a link to each company's Yahoo Finance page.

In [ ]:
# Setup: run from a clone of the repo, locally or in Colab
import os, sys, json
from pathlib import Path

REPO_URL = "https://github.com/roolau/semiconductor-supply-chain-kg-extension.git"   # Colab only: the repo to clone
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("/content/repo").exists():
    !git clone -q {REPO_URL} /content/repo
ROOT = Path("/content/repo") if IN_COLAB else Path.cwd().resolve().parents[1]   # notebooks/ -> extension -> repo
EXT = ROOT / "equity-research-extension"   # this extension: code, reference data, outputs

# The Phase 1 KG (read-only): the folder holding fused_edges.jsonl and canonical_entities.jsonl
KG = Path(os.environ.get("P1_KG_DIR", ROOT / "data"))   # data/ at the repo root
assert (EXT / "src/kg/load.py").exists(), f"extension code not found under {EXT}"
assert (KG / "fused_edges.jsonl").exists(), "Phase 1 KG not found: set P1_KG_DIR to the folder with fused_edges.jsonl"
os.environ["P1_KG_DIR"] = str(KG)
if str(EXT) not in sys.path:
    sys.path.insert(0, str(EXT))

In [ ]:
!pip -q install yfinance

In [ ]:
import importlib; importlib.invalidate_caches()
from src.kg import load, finance
for m in (load, finance):
    importlib.reload(m)
kg = load.load()
tickers = finance.tickers_from_kg(kg)
print(len(tickers), 'tickers')
data = finance.fetch_all(tickers, EXT / 'data/raw/yfinance', pause=3.0)
path = finance.write_snapshot(data, EXT / 'data/processed/finance/yfinance_snapshot.json')
bad = {k: v.get('error') for k, v in data.items() if 'error' in v or not v.get('prices')}
print(path, '| failed or no prices:', bad or 'none')

If some tickers fail (Yahoo sometimes rate-limits), run the last cell again: cached tickers are skipped and only the failed ones are fetched. The site build (`src/kg/site.py`) reads this snapshot.